# Product Query Agent - Single Angent

In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent


load_dotenv()

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### Agent Route

``` python
LLM (Brain) -> Has access to tools
```


## Agent chat
``` python
1. extract product name from question
2. get_product("wireless headphones")
3. analyze the answer: {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."}
4. return human readable answer
```

In [2]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)


# LLM Creation
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature = 0)

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful assistant for an online tech store"
)


In [3]:
def ask_question(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [4]:
ask_question("What is the price of the wireless headphones.")

The price of the wireless headphones is $79.99.


In [5]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}


@tool
def get_review(name: str) -> str:
    """Look up a product review by product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not found. Available: {', '.join(REVIEWS)}"
    return str(r)

In [6]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store"
)

def ask_question_2(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [7]:
ask_question_2("how do people like smart watch")

Based on the product information, people seem to like the smart watch, with a rating of 4.3 out of 5. The product description also highlights its key features, such as tracking heart rate and sleep, having a 5-day battery life, and being water-resistant. Overall, the smart watch appears to be a popular and well-regarded product.


In [8]:
ask_question_2("What is the price of this product?")

The price of the wireless headphones is $79.99.


I've looked up the product information and reviews for the available items: 

1. Smart Watch - $199.99, 4.3 rating, 5-day battery, water-resistant, with 340 reviews and a 3.9 rating.
2. Mechanical Keyboard - $129.00, 4.8 rating, tenkeyless, Cherry MX Brown switches, per-key RGB, with 67 reviews and a 4.8 rating.
3. Laptop Stand - $34.99, 4.5 rating, adjustable aluminium, fits 11-17 inch laptops, folds flat, with 781 reviews and a 4.5 rating.
4. Wireless Headphones - $79.99, 4.6 rating, over-ear Bluetooth, 30-hr battery, active noise cancellation, with 1262 reviews and a 4.6 rating.

Let me know if you need any further assistance!
